In [15]:
import numpy as np
from pyscf import gto, scf
import pandas as pd
from scipy.linalg import eigh

In [ ]:
def hartree_fock(S, H_core, eri, n_electrons, max_iter=50, tol=1e-6):
    """
    Basic restricted Hartree-Fock (RHF) implementation.
    
    Parameters:
        S: overlap matrix
        H_core: core Hamiltonian
        eri: two-electron integrals (4D tensor)
        n_electrons: total number of electrons
    """

    n_basis = S.shape[0]
    n_occ = n_electrons // 2

    # orthogonalization, get eigenvalues and eigenvectors of overlap matrix
    eigvals, eigvecs = eigh(S)
    S_inv_sqrt = eigvecs @ np.diag(eigvals**-0.5) @ eigvecs.T

    # set init guess
    P = np.zeros((n_basis, n_basis))  # density matrix

    E_old = 0

    # main fock guessing
    for iteration in range(max_iter):

        # build matrix
        J = np.einsum('pqrs,rs->pq', eri, P)
        K = np.einsum('prqs,rs->pq', eri, P)
        F = H_core + J - 0.5 * K

        # solve roothaan equations
        F_prime = S_inv_sqrt.T @ F @ S_inv_sqrt
        eps, C_prime = eigh(F_prime)
        C = S_inv_sqrt @ C_prime

        # build new density matrix
        C_occ = C[:, :n_occ]
        P_new = 2 * C_occ @ C_occ.T

        # compute energy
        E = 0.5 * np.sum(P_new * (H_core + F))

        # check convergence
        dE = abs(E - E_old)
        dP = np.linalg.norm(P_new - P)
        if dE < tol and dP < tol:
            break

        # continue if not converged yet (based on tolerance)
        P = P_new
        E_old = E

    return E, C, P

In [11]:
def compare(mol):
    S = mol.intor('int1e_ovlp')
    T = mol.intor('int1e_kin')
    V = mol.intor('int1e_nuc')
    H_core = T + V
    eri = mol.intor('int2e')

    n_electrons = mol.nelectron
    E_nuc = mol.energy_nuc()

    E_elec, C, P = hartree_fock(
        S=S,
        H_core=H_core,
        eri=eri,
        n_electrons=n_electrons
    )

    E_total = E_elec + E_nuc

    mf = scf.RHF(mol)
    E_total_scf = mf.kernel()

    data = {
        'Implementation Results': [E_elec, E_nuc, E_total, C, P], 
        'PySCF Results': [mf.energy_elec()[0], mol.energy_nuc(), E_total_scf, mf.mo_coeff, mf.make_rdm1()]}

    return pd.DataFrame(data, index=['Electronic energy', 'Nuclear Repulsion', 'Total HF Energy', 'MO Coefficients', 'Density Matrix'])

In [20]:
mol = gto.M(
    atom='H 0 0 0; H 0 0 0.74',
    basis='sto-3g',
    unit='Angstrom'
)

mol = gto.M(
    atom="""
    O  0.0000  0.0000  0.0000
    H  0.0000 -0.7570  0.5870
    H  0.0000  0.7570  0.5870 
    """,
    basis="sto-3g",
    charge=0,
    spin=0,
    unit="Angstrom"
)

mol = gto.M(
    atom="O 0 0 0",
    basis="sto-3g",
    spin=2
)

In [21]:
compare(mol)

converged SCF energy = -73.8041502332559


,Implementation Results,PySCF Results
Electronic energy,-73.661817,-73.80415
Nuclear Repulsion,0,0
Total HF Energy,-73.661817,-73.80415
MO Coefficients,"[[0.9950278051062853, -0.2631994835334744, 0.0...","[[0.9950278051062856, -0.26319948353347444, 0...."
Density Matrix,"[[2.1187086021338386, -0.5015066664444745, 0.0...","[[[1.05935430106692, -0.25075333322223753, 0.0..."
